In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import logging
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from src.helper import *
from src.model_sota import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
# EMBEDDED_DATA_PATH = "output/embeddings/wav2vec2/"
OUTPUT_PATH = "output/"
PREDS_PATH = "output/preds/"
MODELS_PATH = "models/"

#model training config
BATCH_SIZE = 16
# EPOCHS = 20
EPOCHS = 3
LEARNING_RATE = 0.0001

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(PREDS_PATH):
    os.mkdir(PREDS_PATH)
if not os.path.exists(MODELS_PATH):
    os.mkdir(MODELS_PATH)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_05_sota_model_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting SOTA modeling...")

### Preparing the data loader

In [6]:
logger.info(f"preparing the dataloader...")

train_ds = AudioFolderDataset(os.path.join(DATA_PATH, "training/"))
test_ds = AudioFolderDataset(os.path.join(DATA_PATH, "testing/"))
holdout_ds = AudioFolderDataset(os.path.join(DATA_PATH, "holdout/"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Inspect one batch
for X, y, path in train_loader:
    logger.info(f"batch input shape: {X.shape}")
    logger.info(f"batch labels shape: {y.shape}")
    break

### Testing SOTA HybridAudioClassifier Model 

It mixes a strong CNN backbone (PANNs-style), residual + squeeze-and-excitation blocks, and a Transformer encoder on top (AST / PaSST style). This hybrid approach is widely used in modern audio classification and often gives top results on benchmarks like AudioSet, ESC-50 and Speech Commands.

In [ ]:
model = HybridAudioClassifier(num_classes=2, use_spec_augment=True).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    print(f"Epoch {epoch}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
            f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    # torch.save(model.state_dict(), f"checkpoint_epoch{epoch}.pth")

[WARN] Failed to load data/deep-detect/dataset/training/fake/file17450.mp3: Error opening 'data/deep-detect/dataset/training/fake/file17450.mp3': File does not exist or is not a regular file (possibly a pipe?).
[WARN] Failed to load data/deep-detect/dataset/training/fake/file19851.mp3: Error opening 'data/deep-detect/dataset/training/fake/file19851.mp3': File does not exist or is not a regular file (possibly a pipe?).


Note: Illegal Audio-MPEG-Header 0x00000000 at offset 50.
Note: Trying to resync...
Note: Hit end of (available) data during resync.


[WARN] Failed to load data/deep-detect/dataset/training/fake/file17407.mp3: Error opening 'data/deep-detect/dataset/training/fake/file17407.mp3': File does not exist or is not a regular file (possibly a pipe?).
[WARN] Failed to load data/deep-detect/dataset/training/fake/file32972.mp3: Error opening 'data/deep-detect/dataset/training/fake/file32972.mp3': File does not exist or is not a regular file (possibly a pipe?).
[WARN] Failed to load data/deep-detect/dataset/training/fake/file30959.mp3: Error opening 'data/deep-detect/dataset/training/fake/file30959.mp3': File does not exist or is not a regular file (possibly a pipe?).
[WARN] Failed to load data/deep-detect/dataset/training/fake/file27839.mp3: Error opening 'data/deep-detect/dataset/training/fake/file27839.mp3': File does not exist or is not a regular file (possibly a pipe?).
[WARN] Failed to load data/deep-detect/dataset/training/fake/file16643.mp3: Error opening 'data/deep-detect/dataset/training/fake/file16643.mp3': File does 

Note: Illegal Audio-MPEG-Header 0x00000000 at offset 50.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
